# 04 Train and Evaluate with W&B

Run the full train-to-evaluation pipeline and log experiment results to Weights & Biases. This notebook logs config values, data sizes, train/validation losses, BLEU, chrF, sample translations, and checkpoint artifacts.

## Colab Setup

Run the next cell first when using Google Colab. It clones the repository, installs project dependencies from `pyproject.toml`, and moves the working directory to the repository root.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/songhahyun/seq2seq_transformer_model.git"
REPO_DIR = Path("/content/seq2seq_transformer_model")

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--branch", "dev", "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", ".[notebooks]"], check=True)
else:
    print("Not running in Colab; skipping clone/install.")

print(f"cwd: {Path.cwd()}")

In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo root: {REPO_ROOT}")

## W&B settings

Set `WANDB_MODE = "offline"` if you want to log locally without syncing immediately. For online logging, put `WANDB_API_KEY` in `.env` or run `wandb login` before executing this notebook.

In [ ]:
from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env", override=False)

PROJECT_NAME = os.getenv("WANDB_PROJECT", "seq2seq-transformer")
ENTITY_NAME = os.getenv("WANDB_ENTITY") or None
RUN_NAME = "experiment_v0.6"
WANDB_MODE = os.getenv("WANDB_MODE", "online")  # "online" or "offline"
LOG_ARTIFACTS = True
NUM_SAMPLE_TRANSLATIONS = 10
WATCH_LOG = None  # None, "gradients", "parameters", or "all"
WATCH_LOG_FREQ = 100

os.environ["WANDB_MODE"] = WANDB_MODE

if WANDB_MODE == "online" and not os.getenv("WANDB_API_KEY"):
    raise RuntimeError("WANDB_API_KEY is missing. Add it to .env, run wandb login, or set WANDB_MODE='offline'.")

print(f"W&B mode: {WANDB_MODE}")
print(f"W&B project: {PROJECT_NAME}")
print(f"W&B entity: {ENTITY_NAME or '(default)'}")
# print(f"W&B API key loaded: {bool(os.getenv('WANDB_API_KEY'))}")

In [ ]:
import torch.optim as optim
import wandb

from src.config import Config
from src.data_pipeline import (
    create_dataloaders,
    extract_pairs,
    load_ko_en_dataset,
    maybe_take_subset,
    prepare_tokenizers,
    set_seed,
    split_pairs,
)
from src.evaluate import generate_predictions, print_sample_translations
from src.metrics import compute_bleu, compute_chrf
from src.model_utils import build_model
from src.train import (
    create_loss_fn,
    create_lr_scheduler,
    save_checkpoint,
    train_one_epoch,
    validate_one_epoch,
)
from src.wandb_experiment import config_for_wandb, log_checkpoint_artifacts

config = Config()
set_seed(config.random_seed)

print(f"device: {config.device}")
print(f"dataset: {config.dataset_name}")

## Initialize W&B run

In [ ]:
if WANDB_MODE == "online":
    wandb.login(key=os.getenv("WANDB_API_KEY"), relogin=False, timeout=60)

run = wandb.init(
    project=PROJECT_NAME,
    entity=ENTITY_NAME,
    name=RUN_NAME,
    config=config_for_wandb(config),
    settings=wandb.Settings(init_timeout=120),
)

run.name

## Load and split dataset

In [ ]:
dataset = load_ko_en_dataset(
    config.dataset_name,
    split=config.train_split,
    hf_token=config.hf_token,
)
pairs = extract_pairs(dataset, src_col="ko", tgt_col="en")

train_pairs, valid_pairs, test_pairs = split_pairs(
    pairs,
    valid_ratio=config.valid_ratio,
    test_ratio=config.test_ratio,
    seed=config.random_seed,
)

train_pairs = maybe_take_subset(train_pairs, config.train_subset_size)
valid_pairs = maybe_take_subset(valid_pairs, config.valid_subset_size)
test_pairs = maybe_take_subset(test_pairs, config.test_subset_size)

wandb.log({
    "data/total_pairs": len(pairs),
    "data/train_pairs": len(train_pairs),
    "data/valid_pairs": len(valid_pairs),
    "data/test_pairs": len(test_pairs),
})

print(f"total pairs: {len(pairs)}")
print(f"train pairs: {len(train_pairs)}")
print(f"valid pairs: {len(valid_pairs)}")
print(f"test pairs : {len(test_pairs)}")

## Prepare tokenizers and dataloaders

In [ ]:
sp_src, sp_tgt = prepare_tokenizers(train_pairs, config)

train_loader, valid_loader, test_loader = create_dataloaders(
    train_pairs,
    valid_pairs,
    test_pairs,
    sp_src,
    sp_tgt,
    config,
)

wandb.config.update(
    {
        "src_actual_vocab_size": sp_src.get_piece_size(),
        "tgt_actual_vocab_size": sp_tgt.get_piece_size(),
        "train_batches": len(train_loader),
        "valid_batches": len(valid_loader),
        "test_batches": len(test_loader),
        "pretokenize_dataset": config.pretokenize_dataset,
        "num_workers": config.num_workers,
        "pin_memory": config.pin_memory,
        "persistent_workers": config.persistent_workers,
    },
    allow_val_change=True,
)

print(f"src vocab size: {sp_src.get_piece_size()}")
print(f"tgt vocab size: {sp_tgt.get_piece_size()}")
print(f"pretokenize dataset: {config.pretokenize_dataset}")
print(f"num workers: {config.num_workers}")
print(f"pin memory: {config.pin_memory}")
print(f"persistent workers: {config.persistent_workers}")


## Build model

In [ ]:
model = build_model(
    config=config,
    src_vocab_size=sp_src.get_piece_size(),
    tgt_vocab_size=sp_tgt.get_piece_size(),
)
optimizer = optim.Adam(model.parameters(), lr=config.lr)
total_training_steps = len(train_loader) * config.num_epochs
scheduler = create_lr_scheduler(optimizer, config, total_training_steps)
criterion = create_loss_fn(config.pad_id)

num_parameters = sum(p.numel() for p in model.parameters())
wandb.config.update(
    {
        "num_parameters": num_parameters,
        "total_training_steps": total_training_steps,
    },
    allow_val_change=True,
)

if scheduler is not None:
    print(
        "lr scheduler: "
        f"{config.lr_scheduler_type} warmup_steps={config.warmup_steps} "
        f"total_steps={total_training_steps}"
    )

if WATCH_LOG is not None:
    wandb.watch(model, log=WATCH_LOG, log_freq=WATCH_LOG_FREQ)

print(f"parameters: {num_parameters:,}")
print(f"dimension: {config.d_model}")
print(f"encoder layers: {config.num_encoder_layers}")
print(f"decoder layers: {config.num_decoder_layers}")
print(f"feedforward dimension: {config.dim_feedforward}")
print(f"dimension: {config.d_model}")
print(f"attention heads: {config.nhead}")
print(f"epochs: {config.num_epochs}")

## Train and log losses

In [ ]:
history = []
best_valid_loss = float("inf")
epochs_without_improvement = 0

for epoch in range(config.num_epochs):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        config.device,
        scheduler=scheduler,
    )
    valid_loss = validate_one_epoch(
        model,
        valid_loader,
        criterion,
        config.device,
    )

    current_lr = optimizer.param_groups[0]["lr"]

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "valid_loss": valid_loss,
        "lr": current_lr,
    })

    wandb.log(
        {
            "epoch": epoch + 1,
            "train/loss": train_loss,
            "valid/loss": valid_loss,
            "train/lr": current_lr,
        },
        step=epoch + 1,
    )

    print(
        f"[Epoch {epoch + 1}/{config.num_epochs}] "
        f"train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f} | "
        f"lr={current_lr:.8f}"
    )

    latest_checkpoint_path = f"{config.checkpoint_dir}/latest.pt"
    save_checkpoint(
        model=model,
        optimizer=optimizer,
        config=config,
        epoch=epoch + 1,
        train_loss=train_loss,
        valid_loss=valid_loss,
        src_vocab_size=sp_src.get_piece_size(),
        tgt_vocab_size=sp_tgt.get_piece_size(),
        path=latest_checkpoint_path,
        scheduler=scheduler,
    )

    improved = valid_loss < best_valid_loss - config.early_stopping_min_delta
    if improved:
        best_valid_loss = valid_loss
        epochs_without_improvement = 0
        best_checkpoint_path = f"{config.checkpoint_dir}/best.pt"
        save_checkpoint(
            model=model,
            optimizer=optimizer,
            config=config,
            epoch=epoch + 1,
            train_loss=train_loss,
            valid_loss=valid_loss,
            src_vocab_size=sp_src.get_piece_size(),
            tgt_vocab_size=sp_tgt.get_piece_size(),
            path=best_checkpoint_path,
            scheduler=scheduler,
        )
        wandb.run.summary["best_valid_loss"] = best_valid_loss
        wandb.run.summary["best_epoch"] = epoch + 1
        print(f"saved best checkpoint: {best_checkpoint_path}")
    else:
        epochs_without_improvement += 1
        print(
            "no validation improvement "
            f"({epochs_without_improvement}/{config.early_stopping_patience})"
        )
        wandb.log(
            {
                "early_stopping/epochs_without_improvement": epochs_without_improvement,
            },
            step=epoch + 1,
        )

        if epochs_without_improvement >= config.early_stopping_patience:
            print(
                "early stopping triggered: "
                f"best_valid_loss={best_valid_loss:.4f}"
            )
            wandb.run.summary["early_stopped"] = True
            wandb.run.summary["stopped_epoch"] = epoch + 1
            break
else:
    wandb.run.summary["early_stopped"] = False

In [ ]:
import matplotlib.pyplot as plt

epochs = [item["epoch"] for item in history]
train_losses = [item["train_loss"] for item in history]
valid_losses = [item["valid_loss"] for item in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

axes[0].plot(epochs, train_losses, marker="o", color="tab:blue")
axes[0].set_title("Train Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, valid_losses, marker="o", color="tab:orange")
axes[1].set_title("Valid Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Evaluate and log metrics

In [ ]:
decode_results = {}
original_decode_strategy = config.decode_strategy
original_beam_size = config.beam_size

for decode_strategy in ("greedy", "beam"):
    config.decode_strategy = decode_strategy
    print(f"\n[Evaluation: {decode_strategy}]")

    source_texts, predictions, references = generate_predictions(
        model=model,
        dataloader=test_loader,
        sp_tgt=sp_tgt,
        config=config,
        device=config.device,
    )

    bleu = compute_bleu(predictions, references)
    chrf = compute_chrf(predictions, references)

    decode_results[decode_strategy] = {
        "source_texts": source_texts,
        "predictions": predictions,
        "references": references,
        "bleu": bleu,
        "chrf": chrf,
    }

    wandb.log({
        f"test/{decode_strategy}/bleu": bleu,
        f"test/{decode_strategy}/chrf": chrf,
    })
    wandb.run.summary[f"test_{decode_strategy}_bleu"] = bleu
    wandb.run.summary[f"test_{decode_strategy}_chrf"] = chrf

    print(f"BLEU: {bleu:.4f}")
    print(f"chrF: {chrf:.4f}")

comparison_table = wandb.Table(columns=["decode_strategy", "beam_size", "bleu", "chrf"])
for decode_strategy, result in decode_results.items():
    comparison_table.add_data(
        decode_strategy,
        config.beam_size if decode_strategy == "beam" else 1,
        result["bleu"],
        result["chrf"],
    )
wandb.log({"test/decoding_comparison": comparison_table})

config.decode_strategy = original_decode_strategy
config.beam_size = original_beam_size

source_texts = decode_results["greedy"]["source_texts"]
predictions = decode_results["greedy"]["predictions"]
references = decode_results["greedy"]["references"]
bleu = decode_results["greedy"]["bleu"]
chrf = decode_results["greedy"]["chrf"]

wandb.log({
    "test/bleu": bleu,
    "test/chrf": chrf,
})
wandb.run.summary["test_bleu"] = bleu
wandb.run.summary["test_chrf"] = chrf

## Log sample translations

In [ ]:
for decode_strategy, result in decode_results.items():
    source_texts = result["source_texts"]
    predictions = result["predictions"]
    references = result["references"]
    sample_count = min(NUM_SAMPLE_TRANSLATIONS, len(predictions))
    sample_table = wandb.Table(columns=["source", "reference", "prediction"])

    for i in range(sample_count):
        sample_table.add_data(source_texts[i], references[i], predictions[i])

    wandb.log({f"samples/{decode_strategy}_translations": sample_table})

    print(f"\n[Sample Translations: {decode_strategy}]")
    print_sample_translations(
        source_texts=source_texts,
        predictions=predictions,
        references=references,
        n=sample_count,
    )

## Log checkpoint artifacts and finish

In [ ]:
if LOG_ARTIFACTS:
    log_checkpoint_artifacts(wandb, config, run.name)

run.finish()